# H2Loop — Embedding Analysis & Failure Analysis

Run `eval.py` first to generate:
- `data/processed/all_embeddings.npy`
- `data/processed/all_labels.json`
- `data/processed/functions.jsonl` (with full labels + embeddings)
- `results/classification_ablation.json`
- `results/retrieval_metrics.json`
- `results/clustering_info.json`
- `results/umap_fused.png`

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
from collections import Counter

DATA_DIR = Path('../data/processed')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

: 

## Load Data

In [ ]:
functions = []
with open(DATA_DIR / 'functions.jsonl') as f:
    for line in f:
        line = line.strip()
        if line:
            functions.append(json.loads(line))

print(f'Loaded {len(functions)} functions')

fused_emb = np.load(str(DATA_DIR / 'all_embeddings.npy'))
with open(DATA_DIR / 'all_labels.json') as f:
    labels = json.load(f)

fn_names = [fn['function_name'] for fn in functions]
print(f'Embedding shape: {fused_emb.shape}')

## Label Distribution

In [ ]:
all_effects = [effect for ls in labels for effect in ls]
dist = Counter(all_effects)
print('Side-effect label distribution:')
for k, v in sorted(dist.items(), key=lambda x: -x[1]):
    print(f'  {k:12s}: {v:4d}  ({100*v/len(functions):.1f}%)')

# Source breakdown
sources = Counter(fn.get('side_effects_source', 'unknown') for fn in functions)
print('\nside_effects_source breakdown:')
for k, v in sources.items():
    print(f'  {k}: {v}')

## UMAP — Fused Embeddings

In [ ]:
import umap

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
coords = reducer.fit_transform(fused_emb)
print('UMAP done. Shape:', coords.shape)

In [ ]:
effect_classes = sorted({lbl for ls in labels for lbl in ls})
colors = cm.Set1(np.linspace(0, 0.8, len(effect_classes)))
color_map = {lbl: colors[i] for i, lbl in enumerate(effect_classes)}

fig, ax = plt.subplots(figsize=(12, 9))

for effect in effect_classes:
    mask = [any(l == effect for l in ls) for ls in labels]
    pts = coords[mask]
    ax.scatter(pts[:, 0], pts[:, 1], label=effect, color=color_map[effect],
               alpha=0.65, s=25, edgecolors='none')

ax.set_title('Fused Embeddings — UMAP colored by side_effect label', fontsize=14)
ax.legend(fontsize=11)
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'umap_fused.png', dpi=150)
plt.show()
print('Saved to results/umap_fused.png')

## UMAP — UniXcoder-only vs Fused (Ablation)

In [ ]:
code_emb = np.array([fn['embedding'] for fn in functions], dtype=np.float32)

reducer_code = umap.UMAP(n_components=2, random_state=42)
coords_code = reducer_code.fit_transform(code_emb)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, coords_plot, title in [
    (axes[0], coords_code, 'UniXcoder-only (768-dim)'),
    (axes[1], coords, 'Fused (776-dim)'),
]:
    for effect in effect_classes:
        mask = [any(l == effect for l in ls) for ls in labels]
        pts = coords_plot[mask]
        ax.scatter(pts[:, 0], pts[:, 1], label=effect, color=color_map[effect],
                   alpha=0.65, s=20, edgecolors='none')
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=10)
    ax.set_xlabel('UMAP-1')
    ax.set_ylabel('UMAP-2')

plt.suptitle('Ablation: UniXcoder-only vs Fused', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'umap_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

## Classification Ablation Results

In [ ]:
with open(RESULTS_DIR / 'classification_ablation.json') as f:
    ablation = json.load(f)

print(f'{"Variant":<30} {"Macro F1":>10} {"Micro F1":>10}')
print('-' * 55)
for variant, res in ablation.items():
    print(f'{variant:<30} {res["macro_f1"]:>10.4f} {res["micro_f1"]:>10.4f}')

## Retrieval Results

In [ ]:
with open(RESULTS_DIR / 'retrieval_metrics.json') as f:
    retrieval = json.load(f)

for k, v in retrieval.items():
    print(f'  {k}: {v:.4f}')

## Failure Analysis — Classification

Finding the class with lowest F1 and example misclassified functions.

In [ ]:
fused_per_class = ablation.get('Fused (776-dim)', {}).get('per_class', {})
if fused_per_class:
    worst = min(fused_per_class, key=lambda c: fused_per_class[c])
    print(f'Worst class: {worst!r} — F1 = {fused_per_class[worst]:.4f}')
    print()
    # Show a few functions with that label
    examples = [fn for fn, ls in zip(functions, labels) if worst in ls][:5]
    for ex in examples:
        print(f"  {ex['function_name']} | source={ex.get('side_effects_source')} | "
              f"labels={ex.get('labels', {}).get('side_effects')}")

    print()
    print('Root cause analysis:')
    print('  - Fewer training examples for this class relative to corpus size')
    print('  - Overlapping call patterns with other classes (e.g., memory inside hardware drivers)')
    print('  - LLM fallback labeling noise for ambiguous functions')

## Failure Analysis — Retrieval

Finding a query that retrieved irrelevant results.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def get_top5(query_idx):
    sims = cosine_similarity(fused_emb[query_idx:query_idx+1], fused_emb)[0]
    sims[query_idx] = -1.0
    return np.argsort(sims)[::-1][:5]

# Find a function labeled 'none' (hardest to retrieve)
none_indices = [i for i, ls in enumerate(labels) if ls == ['none']]
print(f'Functions labeled ["none"]: {len(none_indices)}')

if none_indices:
    q = none_indices[0]
    top5 = get_top5(q)
    print(f'\nQuery: {fn_names[q]!r} — labels: {labels[q]}')
    print('Top-5 retrieved:')
    for rank, idx in enumerate(top5, 1):
        match = '✓' if set(labels[idx]) & set(labels[q]) else '✗'
        print(f'  [{rank}] {match} {fn_names[idx]!r} — labels: {labels[idx]}')
    print()
    print('Root cause: "none" functions are rare in this corpus. Their embeddings sit')
    print('near whichever hardware/io cluster has the most similar naming conventions.')

## Failure Analysis — Clustering

Finding a function that landed in the wrong cluster.

In [ ]:
from sklearn.cluster import KMeans
from collections import Counter

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(fused_emb)

priority = {'hardware': 0, 'io': 1, 'memory': 2, 'none': 3}
dominant = [min(ls, key=lambda x: priority.get(x, 99)) for ls in labels]

# Determine each cluster's dominant label
cluster_dominant = {}
for c in range(3):
    mask = cluster_labels == c
    contents = [dominant[i] for i in range(len(dominant)) if mask[i]]
    cluster_dominant[c] = Counter(contents).most_common(1)[0][0] if contents else '?'

print('Cluster → dominant label:', cluster_dominant)

# Find misclassified functions
misclassified = [
    (fn_names[i], dominant[i], int(cluster_labels[i]), cluster_dominant[cluster_labels[i]])
    for i in range(len(dominant))
    if dominant[i] != cluster_dominant[cluster_labels[i]]
]

print(f'\nMisclassified functions: {len(misclassified)} / {len(functions)}')
print('\nExample misclassified functions (name | true_label | cluster | cluster_dominant):')
for name, true_lbl, cl, cl_dom in misclassified[:10]:
    print(f'  {name:<45} {true_lbl:<12} → cluster {cl} ({cl_dom})')

if misclassified:
    ex_name = misclassified[0][0]
    ex_fn = next(fn for fn in functions if fn['function_name'] == ex_name)
    print(f'\nDetailed failure case: {ex_name!r}')
    print(f'  side_effects: {ex_fn.get("labels", {}).get("side_effects")}')
    print(f'  call_count:   {ex_fn.get("ast_features", {}).get("call_count")}')
    print(f'  side_effects_source: {ex_fn.get("side_effects_source")}')
    print()
    print('Root cause: mixed-signal functions sit on class boundaries in embedding space.')
    print('The fused embedding preserves both signals, but K-means assigns a single centroid.')

## AST Feature Distributions

In [ ]:
import pandas as pd

ast_records = [fn['ast_features'] for fn in functions]
df = pd.DataFrame(ast_records)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='none', alpha=0.8)
    axes[i].set_title(col)
    axes[i].set_xlabel('value')
    axes[i].set_ylabel('count')

plt.suptitle('AST Feature Distributions (484 functions)', fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ast_feature_distributions.png', dpi=150)
plt.show()

print(df.describe().round(2))